# Prueba corta antes de las 90 corridas (3.2.1, paso 1)

Corre **MO-01 con la semilla 4** en las tres liquideces: el sistema, TWAP y VWAP (9 corridas, ≈ 25 min). Sirve para comprobar que todo funciona antes de lanzar los 6 cuadernos: que se leen los escenarios, que cargan los checkpoints y que los archivos salen con el nombre de cada liquidez.

**No** sirve para concluir nada: es una sola corrida por escenario. Los resultados quedan en `corridas_90_prueba/` y no se mezclan con las 90.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-3-2-1-cuadernos, o main si ya se mergeó; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && (git checkout -q -f origin/feature/ps-3-2-1-cuadernos || git checkout -q -f origin/main)
!cd /content/slippage && git log --oneline -1 && grep -c "semilla_inicial" scripts/colab/evaluar_sistema_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Prueba

In [ ]:
# CELDA 10: prueba corta (≈ 25 min)
for ESCENARIO in ('baja', 'media', 'alta'):
    OUT = f'{RESULTS_DIR}/corridas_90_prueba/{ESCENARIO}'
    comun = f'--meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-01 --seeds 1 --semilla-inicial 4 --escenario {ESCENARIO} --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --out-dir {OUT}'
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py {comun} --completar-al-cierre --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --nombre PPO_sistema 2>&1 | grep -v UserWarning")
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/benchmarks_meta_ordenes_ps.py {comun} 2>&1 | grep -v UserWarning")


## Resumen

In [ ]:
# CELDA 11: resumen de la prueba (deben salir 3 filas por escenario, sin errores y con cumplimiento cercano a 1)
import json, glob
for f in sorted(glob.glob(f'{RESULTS_DIR}/corridas_90_prueba/*/*.json')):
    for c in json.load(open(f, encoding='utf-8')).get('corridas', {}).values():
        if 'error' in c:
            print('ERROR', c['estrategia'], c['error'][:120])
        else:
            print(f"{c.get('escenario'):6s} {c['estrategia']:12s} semilla {c['seed']}  costo {c['slippage_bps']:8.2f} bps  cumplimiento {c.get('execution_rate', 1):.3f}")
